# Numerical differences in JAX domain calculations

This notebook holds inputs fixed, isolates solver and interpolation boundaries,
and proves exact original-kernel controls by comparing dtype, shape, and bytes.
Differences are observations for the printed software/device environment.
Original validation is deliberately slow and does not support differentiation.
Temporary TOV tables are generated and removed within the notebook.

In [1]:
import warnings
warnings.filterwarnings("ignore", message="Wswiglal-redir-stdio:.*")
warnings.filterwarnings("ignore", message="fval is not bracketed.*")
import os
import tempfile
from pathlib import Path
from fractions import Fraction
import importlib.metadata
import numpy as np
import jax
import jax.numpy as jnp
from pycbc import conversions, cosmology, transforms
from pycbc.coordinates import base as coordinates
from pycbc.scheme import CPUScheme, JAXScheme

DEVICE = os.environ.get("PYCBC_TEST_SCHEME", "jax:cpu").removeprefix("jax:")
jax.config.update("jax_enable_x64", True)
print({name: importlib.metadata.version(name)
       for name in ("numpy", "scipy", "jax", "astropy", "pykerr")})
with JAXScheme(DEVICE) as context:
    print({"platform": context.jax_device.platform, "x64": jax.config.x64_enabled})


def exact(actual, expected):
    a, b = np.asarray(actual), np.asarray(expected)
    assert a.shape == b.shape and a.dtype == b.dtype
    assert a.tobytes() == b.tobytes()


def original(function, *values, **kwargs):
    with CPUScheme():
        return function(*values, **kwargs)


def evaluate(function, *values, controls=(), **kwargs):
    with JAXScheme(DEVICE, reference_operations=controls) as context:
        inputs = [jax.device_put(np.asarray(value), context.jax_device)
                  for value in values]
        return function(*inputs, **kwargs)


def difference(cpu, device):
    a, b = np.asarray(cpu), np.asarray(device)
    return {"original_dtype": str(a.dtype), "jax_dtype": str(b.dtype),
            "max_abs_difference": float(np.max(abs(a - b))),
            "same_bytes": a.dtype == b.dtype and a.tobytes() == b.tobytes()}

{'numpy': '2.5.2', 'scipy': '1.16.3', 'jax': '0.11.1', 'astropy': '8.0.1', 'pykerr': '0.2.0'}
{'platform': 'cpu', 'x64': True}


## Same mass equation, different numerical solver

The original chirp-mass inverse calls `numpy.roots`, which uses eigenvalues of a
companion matrix. JAX uses an analytic dimensionless cubic solution and one
Newton correction in logarithmic mass ratio. The known-mass/eta inverse uses
an explicit quadratic solution instead of the original root solver.
Their arithmetic sequences differ even in float64. With float32 tensors, the
JAX output also stays float32 while the original cubic solver returns float64.
The checks below retain both original solvers independently.

In [2]:
for dtype in (np.float32, np.float64):
    chirp, mass1 = np.array([16., 1.1], dtype=dtype), np.array([30., 1.4], dtype=dtype)
    cpu = original(conversions.mass2_from_mchirp_mass1, chirp, mass1)
    default = evaluate(conversions.mass2_from_mchirp_mass1, chirp, mass1)
    native = evaluate(conversions.mass2_from_mchirp_mass1, chirp, mass1,
                      controls=("conversions.mass2_from_mchirp_mass1",))
    exact(native, cpu)
    recovered = evaluate(conversions.mchirp_from_mass1_mass2, mass1, np.asarray(default))
    np.testing.assert_allclose(recovered, chirp, rtol=2e-6 if dtype == np.float32 else 1e-13)
    print("chirp inverse", dtype.__name__, np.asarray(cpu), np.asarray(default), difference(cpu, default))
    eta = np.array([.1, .24], dtype=dtype)
    cpu = original(conversions.mass_from_knownmass_eta, mass1, eta)
    native = evaluate(conversions.mass_from_knownmass_eta, mass1, eta,
                      controls=("conversions.mass_from_knownmass_eta",))
    exact(native, cpu)
    print("known-mass inverse", dtype.__name__, "original bytes verified")

chirp inverse float32 [11.7478011   1.14278029] [11.747802   1.1427803] {'original_dtype': 'float64', 'jax_dtype': 'float32', 'max_abs_difference': 6.811165462750068e-07, 'same_bytes': False}
known-mass inverse float32 original bytes verified
chirp inverse float64 [11.7478011   1.14278022] [11.7478011   1.14278022] {'original_dtype': 'float64', 'jax_dtype': 'float64', 'max_abs_difference': 3.552713678800501e-15, 'same_bytes': False}
known-mass inverse float64 original bytes verified


## Linear interpolation: table precision and operation order

NumPy linear interpolation uses double precision. The JAX TOV transform casts
knots and values to the mass precision before interpolating. Its original
validation route explicitly calls the old scalar transform once per vector
entry; ordinary NumPy vector calls keep the original API's errors.

Even with identical float64 tables, NumPy's interpolation kernel multiplies a
precomputed slope by the offset, while JAX divides the offset by the interval
width first. Compilers can fuse the final multiply/add. The next control checks
JAX's operation order and classifies the observed CPU rounding, rather than
assuming all NumPy builds fuse identically.

In [3]:
with tempfile.TemporaryDirectory() as directory:
    table = Path(directory) / "tov.txt"
    np.savetxt(table, [[1., 1000.], [1.2, 600.], [1.6, 200.], [2., 50.]])
    tov = transforms.LambdaFromTOVFile("mass1", "lambda1", table, redshift_mass=False)
    for dtype in (np.float32, np.float64):
        masses = np.array([1.1, 1.4, 1.8], dtype=dtype)
        cpu = np.asarray([original(tov.transform, {"mass1": mass})["lambda1"] for mass in masses])
        with JAXScheme(DEVICE):
            default = tov.transform({"mass1": jnp.asarray(masses)})["lambda1"]
        with JAXScheme(DEVICE, reference_operations=("transforms.LambdaFromTOVFile.transform",)):
            native = tov.transform({"mass1": jnp.asarray(masses)})["lambda1"]
        exact(native, cpu)
        print("TOV", dtype.__name__, np.asarray(cpu), np.asarray(default), difference(cpu, default))

xp, yp = np.array([.1, 1.3]), np.array([1.25, 3.7])
x = np.linspace(.12, 1.28, 33)
cpu = np.interp(x, xp, yp)
slope = (yp[1] - yp[0]) / (xp[1] - xp[0])
unfused = yp[0] + slope * (x - xp[0])
fused = np.asarray([float(Fraction(float(slope)) * Fraction(float(offset))
                          + Fraction(float(yp[0]))) for offset in x - xp[0]])
with JAXScheme(DEVICE):
    args = tuple(jnp.asarray(value) for value in (x, xp, yp))
    device = jnp.interp(*args)
    ratio_first = jax.jit(lambda z, knots, values:
        values[0] + ((z - knots[0]) / (knots[1] - knots[0]))
                    * (values[1] - values[0]))(*args)
exact(device, ratio_first)
print("CPU bins matching separate multiply/add:", int(np.count_nonzero(cpu == unfused)))
print("CPU bins matching fused multiply/add:", int(np.count_nonzero(cpu == fused)))
print("identical float64 tables:", difference(cpu, device))

TOV float32 [799.99995232 400.00002384 125.00001788] [800.      400.00006 125.00002] {'original_dtype': 'float64', 'jax_dtype': 'float32', 'max_abs_difference': 4.76837158203125e-05, 'same_bytes': False}


TOV float64 [800. 400. 125.] [800. 400. 125.] {'original_dtype': 'float64', 'jax_dtype': 'float64', 'max_abs_difference': 0.0, 'same_bytes': True}
CPU bins matching separate multiply/add: 27
CPU bins matching fused multiply/add: 33
identical float64 tables: {'original_dtype': 'float64', 'jax_dtype': 'float64', 'max_abs_difference': 4.440892098500626e-16, 'same_bytes': False}


## Cosmology reuses the original double-precision grids

Astropy constructs the fixed grids on the host; evaluation of supported JAX
arrays uses device interpolation. Default distances are restricted to the
precomputed redshift range, and volumes to the range from redshift 0.001.
The independent native route retains the original interpolation and fallback
behavior. The examples compare the same converter objects and grids.

In [4]:
distance_converter = cosmology.DistToZ(numpoints=256)
distances = np.array([1., 100., 1000., 10000.])
cpu = original(distance_converter, distances)
default = evaluate(distance_converter, distances)
native = evaluate(distance_converter, distances,
                  controls=("cosmology.DistToZ.get_redshift",))
exact(native, cpu)
print("distance interpolation:", difference(cpu, default))

volume_converter = cosmology.ComovingVolInterpolator("redshift", numpoints=64)
volume_converter.setup_interpolant()
volumes = volume_converter.cosmology.comoving_volume(np.array([.01, .5, 2.])).value
cpu = original(volume_converter, volumes)
native = evaluate(volume_converter, volumes, controls=("cosmology",))
exact(native, cpu)
print("volume interpolation: original bytes verified")

distance interpolation: {'original_dtype': 'float64', 'jax_dtype': 'float64', 'max_abs_difference': 0.0, 'same_bytes': True}


volume interpolation: original bytes verified


## Elementary functions and independent composition controls

Coordinate and log transforms use different NumPy and XLA elementary-function
kernels. Here a mass solver, coordinate conversion, and log transform are
independently selectable. Enabling all three original stages restores the
whole composition's original bytes, while retaining JAX storage between calls.

In [5]:
log = transforms.Log("rho", "logrho")
def calculation(chirp, mass1):
    mass2 = conversions.mass2_from_mchirp_mass1(chirp, mass1)
    rho, _, _ = coordinates.cartesian_to_spherical(mass1, mass2, mass1 * 0)
    return log.transform({"rho": rho})["logrho"]

chirp, mass1 = np.array([16., 1.1], dtype=np.float32), np.array([30., 1.4], dtype=np.float32)
cpu = original(calculation, chirp, mass1)
controls = ("conversions.mass2_from_mchirp_mass1",
            "coordinates.cartesian_to_spherical", "transforms.Log.transform")
for selected in ((), controls[:1], controls[:2], controls):
    result = evaluate(calculation, chirp, mass1, controls=selected)
    print({"controls": selected, **difference(cpu, result)})
exact(evaluate(calculation, chirp, mass1, controls=controls), cpu)

{'controls': (), 'original_dtype': 'float64', 'jax_dtype': 'float32', 'max_abs_difference': 9.290335523104432e-08, 'same_bytes': False}
{'controls': ('conversions.mass2_from_mchirp_mass1',), 'original_dtype': 'float64', 'jax_dtype': 'float32', 'max_abs_difference': 9.290335523104432e-08, 'same_bytes': False}
{'controls': ('conversions.mass2_from_mchirp_mass1', 'coordinates.cartesian_to_spherical'), 'original_dtype': 'float64', 'jax_dtype': 'float64', 'max_abs_difference': 0.0, 'same_bytes': True}
{'controls': ('conversions.mass2_from_mchirp_mass1', 'coordinates.cartesian_to_spherical', 'transforms.Log.transform'), 'original_dtype': 'float64', 'jax_dtype': 'float64', 'max_abs_difference': 0.0, 'same_bytes': True}


## QNM spline data and original dependency errors

QNM conversions reuse the installed pykerr spline data. JAX tables use the
tensor precision and Horner evaluation; original SciPy evaluation uses double
precision. The original control calls the actual original public API. If an
installed original dependency fails (for example, pykerr using removed
`numpy.float`), the control reproduces the same exception instead of replacing
the implementation or repairing unrelated code.

In [6]:
mass, spin = np.array([30., 60.]), np.array([.1, .2])
default = evaluate(conversions.get_lm_f0tau, mass, spin, l=2, m=2)
print("default QNM frequency:", np.asarray(default[0]))
try:
    cpu = original(conversions.get_lm_f0tau, mass, spin, l=2, m=2)
except AttributeError as error:
    with JAXScheme(DEVICE, reference_operations=("conversions.get_lm_f0tau",)):
        try:
            conversions.get_lm_f0tau(jnp.asarray(mass), jnp.asarray(spin), 2, 2)
        except type(error) as reference_error:
            assert str(reference_error) == str(error)
            print("original dependency exception verified:", type(error).__name__)
        else:
            raise AssertionError("original dependency exception was not reproduced")
else:
    native = evaluate(conversions.get_lm_f0tau, mass, spin, l=2, m=2,
                      controls=("conversions.get_lm_f0tau",))
    for actual, expected in zip(native, cpu):
        exact(actual, expected)
    print("QNM original bytes verified")

default QNM frequency: [416.85017781 216.57203267]


original dependency exception verified: AttributeError


## Differentiation belongs to the default path

Concrete original controls transfer to host and cannot participate in JAX
tracing. Default logit formulas retain differentiation and compiled valid-input
support; eager invalid inputs still raise. Compiled logit callers must satisfy
the configured bounds.

In [7]:
logit = transforms.Logit("x", "logitx")
with JAXScheme(DEVICE):
    x = jnp.array([.1, .3, .7])
    loss = lambda value: jnp.sum(logit.transform({"x": value})["logitx"])
    gradient = jax.jit(jax.grad(loss))(x)
    np.testing.assert_allclose(gradient, 1 / (np.asarray(x) * (1 - np.asarray(x))), rtol=1e-13)
    jacobian = jax.jit(lambda value: logit.jacobian({"x": value}))(x)
    np.testing.assert_allclose(gradient, jacobian, rtol=1e-13)
print("compiled default logit gradient verified")

compiled default logit gradient verified


## References

- [NumPy root-solver algorithm](https://numpy.org/doc/stable/reference/generated/numpy.roots.html)
- [NumPy linear interpolation implementation](https://github.com/numpy/numpy/blob/v2.5.2/numpy/_core/src/multiarray/compiled_base.c)
- [JAX linear interpolation implementation](https://github.com/jax-ml/jax/blob/jax-v0.11.1/jax/_src/numpy/lax_numpy.py)

The original controls preserve native dtypes and reject precision truncation if
JAX double precision is disabled. Byte equality validates the configured
original implementation on held inputs; it is not a claim that default device
math always reproduces all original bits.